In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, Sequential
from ultralytics import YOLO

train = tf.keras.utils.image_dataset_from_directory('dataset/train', image_size=(224,224), batch_size=32)
test = tf.keras.utils.image_dataset_from_directory('dataset/test', image_size=(224,224), batch_size=32)

base = tf.keras.applications.ResNet50(weights='imagenet', include_top=False, input_shape=(224,224,3))
base.trainable = False

model = Sequential([
    base,
    layers.GlobalAveragePooling2D(),
    layers.Dense(128, activation='relu'),
    layers.Dense(len(train.class_names), activation='softmax')
])

model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.fit(train, validation_data=test, epochs=5)

base.trainable = True
for layer in base.layers[:-20]:
    layer.trainable = False

model.compile(optimizer=tf.keras.optimizers.Adam(1e-5), loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.fit(train, validation_data=test, epochs=3)

model.save('retailmind_classifier.keras')

yolo = YOLO('yolo11n.pt')
yolo.predict('product.jpg', save=True, conf=0.25)